# Контрольная работа по темам 1–4

**Темы:** Введение в RecSys · Данные и постановка задачи · Коллаборативная фильтрация · Матричная факторизация

**Длительность:** 90 минут · **Максимум:** 100 баллов

## Часть 1. Теория (40 баллов)

### Блок А. Короткие вопросы (по 3 балла, всего 15)

**А1.** Пользовательская активность в музыкальном сервисе: события `play`, `skip`, `like`, `add_to_playlist`. Какие из них — implicit, какие — explicit? Приведите пример, когда `play` может быть **шумом**, а не позитивным сигналом.

**А2.** Матрица взаимодействий MovieLens 1M: 6 040 пользователей, 3 706 фильмов, ~1 000 000 оценок. Вычислите sparsity. Почему для реальных e-commerce датасетов значение меньше?

**А3.** Объясните, почему `random split` даёт **утечку** будущего в прошлое. Какой split вы выберете, если у логов есть `timestamp`, и как проверите отсутствие утечки одной строкой кода?

**А4.** В чём отличие **item-based** CF от **user-based** CF с точки зрения:
- (а) вычислительной сложности построения матрицы similarity;
- (б) частоты пересчёта в проде;
- (в) интерпретируемости рекомендации.

**А5.** Почему `np.linalg.svd(R)` на разреженной матрице рейтингов с заполненными нулями — **плохая идея**? Какая постановка задачи у FunkSVD решает эту проблему?


### А1.
*   **Implicit (неявные) сигналы:** `play`, `skip`. Пользователь просто потребляет контент, мы лишь косвенно делаем вывод о его предпочтениях по его поведению.
*   **Explicit (явные) сигналы:** `like`, `add_to_playlist`. Пользователь совершает целенаправленное действие, прямо заявляя о своем отношении к треку (в некоторых классификациях добавление в плейлист называют "strong implicit", но по сути это явный позитивный сигнал).
*   **Когда `play` - это шум:**
    1. **Автовоспроизведение (Autoplay):** Пользователь уснул, отошел от устройства или свернул приложение, а треки продолжают играть друг за другом.
    2. **Мисклик (Случайное нажатие):** Пользователь нажал на трек, понял, что ошибся, и переключил его через пару секунд. Если система логирует факт `play` без учета длительности прослушивания (duration), этот сигнал станет ложноположительным шумом.

### А2.
Формула разреженности (sparsity) - это доля пустых ячеек в матрице взаимодействий:
$$Sparsity = 1 - \frac{\text{Кол-во оценок}}{\text{Кол-во пользователей} \times \text{Кол-во фильмов}}$$
$$Sparsity = 1 - \frac{1\,000\,000}{6\,040 \times 3\,706} = 1 - \frac{1\,000\,000}{22\,384\,240} = 1 - 0.04467 \approx 0.9553$$
**Ответ:** Sparsity $\approx 95.5\%$ (или плотность/density $\approx 4.5\%$).

**Спрятанный подвох:** Для реальных e-commerce датасетов значение **sparsity НЕ меньше, оно значительно БОЛЬШЕ** (обычно $>99.9\%$). Вероятно, в вопросе имелась в виду **плотность (density)**, вот она действительно меньше.
**Почему плотность в e-commerce ниже (а sparsity выше):**
В e-commerce каталоги товаров гигантские (миллионы SKU), а средний пользователь покупает всего несколько товаров в год. В кино (особенно на MovieLens) каталог ограничен, а пользователи смотрят фильмы и ставят оценки гораздо чаще, чем покупают, например, холодильники.

### А3.
*   **Почему `random split` даёт утечку:** Случайное разбиение игнорирует время. Действия пользователя за пятницу могут попасть в `train`, а за среду - в `test`. В результате модель "заглядывает в будущее", обучаясь на паттернах, которых еще не существовало в момент совершения тестового действия (например, модель узнает, что пользователь посмотрел сиквел, чтобы предсказать просмотр первой части). Это завышает метрики в офлайне, но проваливается в проде.
*   **Какой split выбрать:** **Global Time-based split** (глобальное разбиение по времени). Например, все логи до 1 октября - в `train`, всё, что после - в `test` (Out-of-time validation).
*   **Проверка отсутствия утечки одной строкой кода:**
    ```python
    assert train['timestamp'].max() <= test['timestamp'].min(), "Data Leakage detected!"
    ```

### А4
*   **(а) Вычислительная сложность:**
    В большинстве сервисов количество пользователей значительно превышает количество товаров ($U \gg I$). Матрица Item-based имеет размер $I \times I$, а User-based - $U \times U$. Соответственно, строить и хранить матрицу схожести для **Item-based сильно дешевле и быстрее** ($O(I^2)$ против $O(U^2)$).
*   **(б) Частота пересчёта в проде:**
    **User-based** требует очень частого пересчета (в идеале real-time), так как вкусы пользователя и его профиль меняются с каждым новым взаимодействием.
    **Item-based** более стабилен: схожесть между двумя фильмами или товарами почти не меняется со временем. Матрицу item-to-item можно пересчитывать редко (раз в сутки/неделю) в batch-режиме.
*   **(в) Интерпретируемость:**
    **Item-based** обладает высокой интерпретируемостью (легко объяснить юзеру: *"Мы рекомендуем вам этот фильм, потому что вы посмотрели Терминатор"*).
    **User-based** объяснить сложно (*"Мы рекомендуем это, потому что пользователи user42 и user84, с которыми у вас высокая косинусная близость, посмотрели это"* - для клиента это звучит пугающе и непонятно).

### А5
*   **Почему SVD на матрице с нулями - плохая идея:**
    1. **Математическая ошибка:** Заполняя пропуски нулями, мы говорим алгоритму: *"Пользователь посмотрел этот фильм и поставил ему оценку 0 (или сильно ненавидит его)"*. Но на деле пустое значение значит лишь то, что пользователь фильм **не видел**. Алгоритм выучит ложные негативные сигналы.
    2. **Вычислительная катастрофа:** Стандартный алгоритм SVD требует плотной матрицы. Вычисление точного SVD на матрице размером даже $100к \times 100к$ (где $99.9\%$ - искусственные нули) потребует огромного объема оперативной памяти (RAM) и гигантских затрат времени ($O(N^3)$), выполняя вычисления над пустышками.
*   **Как FunkSVD решает проблему:**
    FunkSVD (и его вариации) меняет постановку задачи с точного алгебраического разложения на **задачу машинного обучения (оптимизации)**.
    Модель обучается минимизировать ошибку (Loss) **только на тех ячейках, где оценки (взаимодействия) действительно существуют** (observed ratings), игнорируя пропуски. Это делается с помощью градиентного спуска (SGD) или ALS.
    Формула оптимизации строится только по известным $r_{ui}$:
    $$ \min_{p,q} \sum_{u,i \in Observed} (r_{ui} - p_u \cdot q_i^T)^2 + \lambda(||p_u||^2 + ||q_i||^2) $$
    Это решает обе проблемы: мы не смешиваем "не видел" с "не нравится", и вычисления идут быстро, так как считаются только для $\sim 4.5\%$ заполненных данных.

### Блок B. Развёрнутый вопрос — на выбор один (10 баллов)

**B1.** Опишите алгоритм **weighted ALS** для implicit feedback (Hu, Koren, Volinsky, 2008). Приведите:
- функцию потерь с confidence $c_{ui}$;
- аналитическое решение для $\mathbf p_u$ на шаге обновления;
- почему именно confidence, а не бинарные веса, работает лучше для просмотров/кликов.

**B2.** Что такое **popularity bias** и **filter bubble**? Как они возникают из обучения на логах предыдущей версии рекомендатора? Какие две конкретные метрики помогут их задетектировать, и как они считаются?


### B1. Алгоритм Weighted ALS (или iALS - implicit ALS)

Классический стандарт для работы с неявным фидбеком. Его главная идея заключается в разделении сырых данных о взаимодействиях (кликах/просмотрах) на **бинарное предпочтение** и **уровень уверенности** в этом предпочтении.

### **1. Функция потерь с confidence ($c_{ui}$)**

Обозначим сырое количество взаимодействий (например, число прослушиваний трека или минут просмотра) пользователя $u$ с айтемом $i$ как $r_{ui}$.

Из него мы формируем две новые переменные:
1.  **Бинарное предпочтение ($y_{ui}$):**
    $y_{ui} = 1$, если $r_{ui} > 0$ (контакт был).
    $y_{ui} = 0$, если $r_{ui} = 0$ (контакта не было).
2.  **Уверенность (confidence, $c_{ui}$):**
    $c_{ui} = 1 + \alpha \cdot r_{ui}$, где $\alpha$ - гиперпараметр темпа роста уверенности (обычно подбирается на кросс-валидации).

Функция потерь (Loss) принимает следующий вид:
$$ \min_{\mathbf{p}, \mathbf{q}} \sum_{u,i} c_{ui} (y_{ui} - \mathbf{p}_u^T \mathbf{q}_i)^2 + \lambda \left( \sum_u ||\mathbf{p}_u||^2 + \sum_i ||\mathbf{q}_i||^2 \right) $$

**Где:**
*   $\mathbf{p}_u$ - вектор скрытых признаков (эмбеддинг) пользователя.
*   $\mathbf{q}_i$ - вектор скрытых признаков айтема.
*   $\lambda$ - коэффициент L2-регуляризации для предотвращения переобучения.
*   Суммирование $\sum_{u,i}$ идет вообще по **всем** парам (даже где $y_{ui}=0$), что и делает этот алгоритм уникальным по сравнению с обычным SVD.

### **2. Аналитическое решение для $\mathbf{p}_u$ на шаге обновления**

Поскольку это **Alternating** Least Squares (ALS), мы фиксируем векторы айтемов $\mathbf{q}_i$ и ищем оптимальные $\mathbf{p}_u$. Функция потерь становится квадратичной относительно $\mathbf{p}_u$, что позволяет взять производную, приравнять к нулю и получить точное аналитическое решение.

В матричном виде решение для одного пользователя $u$ выглядит так:

$$ \mathbf{p}_u = (\mathbf{Q}^T \mathbf{C}^u \mathbf{Q} + \lambda \mathbf{I})^{-1} \mathbf{Q}^T \mathbf{C}^u \mathbf{y}_u $$

**Расшифровка матриц:**
*   $\mathbf{Q}$ - матрица эмбеддингов всех айтемов размера $(N_{items} \times K)$, где $K$ - размерность эмбеддинга.
*   $\mathbf{C}^u$ - диагональная матрица размера $(N_{items} \times N_{items})$, где на главной диагонали стоят значения $c_{ui}$ для конкретного пользователя $u$.
*   $\mathbf{y}_u$ - вектор-столбец бинарных предпочтений пользователя $u$ размера $(N_{items} \times 1)$.
*   $\mathbf{I}$ - единичная матрица $(K \times K)$.

**Важный вычислительный трюк авторов статьи:**
Считать $\mathbf{Q}^T \mathbf{C}^u \mathbf{Q}$ для каждого юзера невозможно, так как это умножение огромных матриц. Но можно переписать это как:
$$ \mathbf{Q}^T \mathbf{C}^u \mathbf{Q} = \mathbf{Q}^T \mathbf{Q} + \mathbf{Q}^T (\mathbf{C}^u - \mathbf{I}) \mathbf{Q} $$
Матрица $\mathbf{Q}^T \mathbf{Q}$ считается **один раз** для всех пользователей (она не зависит от $u$). А матрица $(\mathbf{C}^u - \mathbf{I})$ имеет ненулевые элементы **только там, где пользователь взаимодействовал с айтемом**. Это снижает вычислительную сложность с $O(N_{items} \cdot K^2)$ до $O(N_{interacted\_items} \cdot K^2)$, позволяя алгоритму работать в проде невероятно быстро.

### **3. Почему confidence работает лучше, чем бинарные веса (1 и 0)**

Неявный фидбек (клики, просмотры, время на странице) по своей природе **очень зашумлен**. В нём нет явного ответа, понравился ли товар пользователю.

1.  **Проблема бинарных весов:**
    Если мы используем только бинарные 0 и 1, мы приравниваем "мисклик" (пользователь зашел в трек и переключил через 2 секунды) к "любимому треку" (пользователь послушал его 50 раз за месяц). И там, и там $y_{ui}=1$.
2.  **Как это решает confidence ($c_{ui}$):**
    Мы говорим модели: "Мы предполагаем, что всё, на что кликнул пользователь, ему нравится ($y_{ui}=1$). Но мы **уверены** в этом пропорционально тому, сколько раз или как долго он это потреблял".
    * Если прослушиваний $r=1$, то $c = 1 + \alpha$. Уверенность низкая. Модель не будет сильно штрафоваться, если предскажет, что трек юзеру не нравится.
    * Если прослушиваний $r=50$, то $c = 1 + 50\alpha$. Уверенность колоссальная. Модель бросит все силы на то, чтобы предсказать вектор $\mathbf{p}_u$ так, чтобы скалярное произведение $\mathbf{p}_u^T \mathbf{q}_i$ было максимально близко к 1, иначе она получит гигантский штраф в функцию потерь.

Точно так же это работает и для пропусков ($r=0, y=0$): алгоритм учитывает негативный сигнал от непросмотренных фильмов с базовой уверенностью $c=1$, предполагая, что большинство из них юзеру действительно не интересны, но не исключая, что он просто про них не знает.

### Блок C. Разбор ошибки (15 баллов)

Коллега пишет baseline `most_popular`:

```python
def most_popular(df: pd.DataFrame, k: int = 10) -> list[int]:
    counts = df.groupby("item_id").size()
    top = counts.sort_values(ascending=False).head(k)
    return top.index.tolist()

# использование
top_k = most_popular(df_full)  # df_full = train + test
precision = evaluate(top_k, test_df)
```

Найдите **минимум 3 ошибки**: концептуальные и практические. Для каждой — объясните, почему это проблема, и как исправить. Исправленный код приводить не обязательно (это задание на разбор).


### Ошибка 1. Утечка данных (Data Leakage) при расчете популярности
*   **Где в коде:** `top_k = most_popular(df_full) # df_full = train + test`
*   **Почему это проблема (концептуальная):** Коллега передает объединенный датасет (и тренировочный, и тестовый) для подсчета топа. Это означает, что модель «подглядывает в будущее». Она формирует список популярных товаров, опираясь на те самые взаимодействия из `test_df`, которые она должна предсказывать. Из-за этого метрика `precision` будет искусственно завышена в офлайн-оценке, но в проде модель покажет результаты хуже.
*   **Как исправить:** Бейзлайны, как и любые ML-модели, должны обучаться (считать статистики) **строго на `train` выборке**. Нужно передавать `df_train`.

### Ошибка 2. Подсчет сырых взаимодействий вместо уникальных пользователей
*   **Где в коде:** `counts = df.groupby("item_id").size()`
*   **Почему это проблема (концептуальная/практическая):** Метод `.size()` считает *абсолютно все* строки. Если один пользователь (или бот) случайно кликнет на товар 5000 раз или послушает один трек на повторе неделю, этот айтем взлетит в топ рекомендаций. Популярность (особенно для холодного старта) должна измерять **широту аудитории**, а не суммарный объем кликов, который подвержен выбросам.
*   **Как исправить:** Нужно считать количество *уникальных пользователей*, взаимодействовавших с товаром. Вместо `.size()` следует использовать `.nunique()` по колонке с ID пользователя:
    `df.groupby("item_id")["user_id"].nunique()`.

### Ошибка 3. Отсутствие фильтрации "просмотренного" (Seen items filtering)
*   **Где в коде:** Алгоритм возвращает статический глобальный список `top_k`, который применяется ко всем юзерам (так как возвращается просто `list[int]`).
*   **Почему это проблема (концептуальная):** В большинстве e-commerce и VOD-сервисов (кино) не принято рекомендовать то, что пользователь *уже купил или посмотрел* в период `train`. Если самый популярный фильм - "Аватар", и юзер посмотрел его в трейне, он попадет к нему в предсказания, но в `test` (будущем) юзер его смотреть уже не будет. Это рушит Precision.
*   **Как исправить:** Предсказания должны формироваться **индивидуально** для каждого юзера. Нужно взять глобальный топ-N (где N с запасом больше K), вычесть из него те айтемы, которые конкретный юзер уже потреблял в `train`, и из оставшихся вернуть топ-K.

### Ошибка 4. Техническая проблема с функцией evaluate
*   **Где в коде:** `evaluate(top_k, test_df)`
*   **Почему это проблема (практическая):** Метрики типа Precision@K считаются на уровне **каждого пользователя** (сравнивается список предсказанных айтемов для юзера со списком его реальных взаимодействий в тесте, а затем результат усредняется). Коллега передает функции оценки просто *один список из 10 чисел*. Функция `evaluate` либо упадет с ошибкой, либо, если она написана плохо, попытается сопоставить этот один список со всем `test_df`, что является алгоритмическим бредом.
*   **Как исправить:** Функция предсказания должна возвращать структуру, где предсказания привязаны к пользователям (например, `dict` вида `{user_id: [item_1, ..., item_10]}` или `pd.DataFrame`), чтобы `evaluate` мог корректно сопоставить предсказания с таргетами конкретного человека.

## Часть 2. Практика (60 баллов)

Во всех задачах можно пользоваться `numpy`, `pandas`, `scipy.sparse`, `scikit-learn`. **Нельзя** использовать готовые реализации RecSys (`implicit`, `surprise`, `LightFM`).

Входные данные (общие для задач 2.1–2.3):

```python
# логи: DataFrame columns = ["user_id", "item_id", "timestamp", "rating"]
# rating in [1..5], timestamp — unix seconds
```


### Задача 2.1 — Подготовка данных (15 баллов)

Реализуйте функцию:

```python
def prepare_split(
    logs: pd.DataFrame,
    k_core: int = 5,
    test_frac: float = 0.2,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    1) k-core фильтрация: оставить только пользователей и объекты
       с >= k_core взаимодействий (итеративно, пока условие не стабилизируется).
    2) Глобальный временной split: нижние (1 - test_frac) по timestamp -> train,
       верхние test_frac -> test.
    3) Из test удалить пользователей и объекты, которых нет в train.
    4) Проверить: train.timestamp.max() < test.timestamp.min().
    Вернуть (train, test).
    """
```

In [1]:
import pandas as pd

def prepare_split(
    logs: pd.DataFrame,
    k_core: int = 5,
    test_frac: float = 0.2,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    1) k-core фильтрация: оставить только пользователей и объекты
       с >= k_core взаимодействий (итеративно).
    2) Глобальный временной split: нижние (1 - test_frac) по timestamp -> train,
       верхние test_frac -> test.
    3) Из test удалить пользователей и объекты, которых нет в train.
    4) Проверить отсутствие временной утечки (строгое неравенство времени).
    """

    # 1. Итеративная k-core фильтрация
    df = logs.copy()
    while True:
        start_len = len(df)

        # Подсчет взаимодействий для каждого пользователя и объекта
        user_counts = df['user_id'].value_counts()
        item_counts = df['item_id'].value_counts()

        # Оставляем только тех, кто прошел порог
        valid_users = user_counts[user_counts >= k_core].index
        valid_items = item_counts[item_counts >= k_core].index

        # Фильтрация датафрейма
        df = df[df['user_id'].isin(valid_users) & df['item_id'].isin(valid_items)]

        # Если размер датафрейма не изменился, значит фильтрация сошлась
        if len(df) == start_len:
            break

    # 2. Глобальный временной split
    # Сортируем данные по времени
    df = df.sort_values('timestamp').reset_index(drop=True)

    # Находим пороговое время на границе test_frac
    split_idx = int(len(df) * (1 - test_frac))
    split_time = df.loc[split_idx, 'timestamp']

    # Для гарантии строгого неравенства train.max() < test.min(),
    # используем пороговое значение времени, а не просто срез по индексу.
    # (Это решает проблему дубликатов timestamp на границе разреза)
    train = df[df['timestamp'] < split_time].copy()
    test = df[df['timestamp'] >= split_time].copy()

    # 3. Фильтрация холодного старта в тесте (удаление неизвестных из test)
    known_users = set(train['user_id'])
    known_items = set(train['item_id'])

    test = test[
        test['user_id'].isin(known_users) &
        test['item_id'].isin(known_items)
    ].copy()

    # 4. Проверка отсутствия утечки будущего в прошлое
    if not train.empty and not test.empty:
        assert train['timestamp'].max() < test['timestamp'].min(), \
            f"Time leakage detected! Train max: {train['timestamp'].max()}, Test min: {test['timestamp'].min()}"

    return train, test

### Задача 2.2 — Item-based kNN на cosine similarity (20 баллов)

Реализуйте класс:

```python
class ItemKNN:
    def __init__(self, k: int = 50, shrink: float = 10.0):
        self.k = k
        self.shrink = shrink

    def fit(self, train: pd.DataFrame) -> "ItemKNN":
        """
        Построить sparse user-item матрицу R (scipy.sparse.csr_matrix),
        посчитать cosine similarity между объектами с shrinkage:
            sim'(i, j) = (|U_i ∩ U_j| / (|U_i ∩ U_j| + shrink)) * cos(i, j).
        Оставить top-k соседей для каждого объекта.
        """

    def recommend(self, user_id: int, n: int = 10) -> list[int]:
        """
        Для user_id посчитать score для каждого item как сумму
        sim(i, j) * r_uj по j из истории пользователя.
        Исключить уже увиденные объекты. Вернуть top-n item_id по score.
        """
```

Требования:
- матрица similarity должна строиться через **разреженные** операции (без плотного $O(n^2)$ хранения в памяти при $n > 10^4$);
- для косинуса использовать нормировку по L2 **колонок** матрицы $R$;
- shrinkage применяется поэлементно к финальной similarity.


In [2]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

class ItemKNN:
    def __init__(self, k: int = 50, shrink: float = 10.0):
        self.k = k
        self.shrink = shrink

    def fit(self, train: pd.DataFrame) -> "ItemKNN":
        # Создание маппингов (внутренние индексы)
        unique_users = train['user_id'].unique()
        unique_items = train['item_id'].unique()

        self.user2idx = {u: i for i, u in enumerate(unique_users)}
        self.idx2item = {i: v for i, v in enumerate(unique_items)}
        self.item2idx = {v: i for i, v in self.idx2item.items()}

        n_users = len(unique_users)
        n_items = len(unique_items)

        # Построение разреженной матрицы R (user x item)
        row = train['user_id'].map(self.user2idx).values
        col = train['item_id'].map(self.item2idx).values
        data = train['rating'].values.astype(float)

        self.R = csr_matrix((data, (row, col)), shape=(n_users, n_items))

        # Нормировка столбцов и косинусное сходство (sparse-матричное умножение)
        # normalize по axis=0 нормирует каждый столбец (item) по L2-норме
        R_norm = normalize(self.R, norm='l2', axis=0)
        # Получаем матрицу Item x Item (Cosine Similarity)
        cos_sim = R_norm.T.dot(R_norm)

        # Матрица пересечений (Co-occurrence)
        # Бьем матрицу R в бинарную (1 там, где было взаимодействие)
        R_bin = self.R.copy()
        R_bin.data = np.ones_like(R_bin.data)
        # Получаем матрицу Item x Item с размером пересечения аудиторий |U_i ∩ U_j|
        cooccur = R_bin.T.dot(R_bin)

        # Shrinkage
        # Подготавливаем матрицу shrink_factor = |U_i ∩ U_j| / (|U_i ∩ U_j| + shrink)
        shrink_factor = cooccur.copy()
        shrink_factor.data = shrink_factor.data / (shrink_factor.data + self.shrink)

        # Применяем shrinkage поэлементно (благодаря multiply в scipy.sparse)
        sim = cos_sim.multiply(shrink_factor)

        # Фильтрация top-k соседей для каждого объекта
        data_topk = []
        indices_topk = []
        indptr_topk = [0]

        for i in range(n_items):
            start = sim.indptr[i]
            end = sim.indptr[i+1]
            row_data = sim.data[start:end]
            row_indices = sim.indices[start:end]

            # Удаляем similarity товара с самим собой (главную диагональ)
            mask = (row_indices != i)
            row_data = row_data[mask]
            row_indices = row_indices[mask]

            # Оставляем только top-k значений с помощью argpartition (работает за O(N))
            if len(row_data) > self.k:
                top_idx = np.argpartition(-row_data, self.k - 1)[:self.k]
                data_topk.extend(row_data[top_idx])
                indices_topk.extend(row_indices[top_idx])
            else:
                data_topk.extend(row_data)
                indices_topk.extend(row_indices)

            indptr_topk.append(len(data_topk))

        # Сохраняем финальную разреженную матрицу similarity
        self.sim_matrix = csr_matrix((data_topk, indices_topk, indptr_topk), shape=(n_items, n_items))

        return self

    def recommend(self, user_id: int, n: int = 10) -> list[int]:
        # Защита от холодного старта (юзера не было в трейне)
        if user_id not in self.user2idx:
            return []

        u_idx = self.user2idx[user_id]

        # Вектор истории пользователя (1 x n_items)
        user_history = self.R[u_idx]

        # Векторное перемножение: scores = history @ sim_matrix
        scores = user_history.dot(self.sim_matrix).toarray().flatten()

        # Исключаем просмотренные товары
        seen_indices = user_history.indices
        scores[seen_indices] = -np.inf

        # Берем top-n индексов с самыми высокими score
        if len(scores) <= n:
            top_indices = np.argsort(-scores)
        else:
            # Быстрый поиск top-n и последующая сортировка только их
            top_indices_unsorted = np.argpartition(-scores, n - 1)[:n]
            top_indices = top_indices_unsorted[np.argsort(-scores[top_indices_unsorted])]

        # Отфильтровываем товары, если у юзера больше нечего рекомендовать
        # (чтобы не возвращать товары со score = -inf)
        top_indices = [idx for idx in top_indices if scores[idx] > -np.inf][:n]

        return [self.idx2item[idx] for idx in top_indices]

### Задача 2.3 — FunkSVD с bias и ранней остановкой (25 баллов)

Реализуйте матричную факторизацию SGD-методом:

$$
\hat r_{ui} = \mu + b_u + b_i + \mathbf p_u^\top \mathbf q_i
$$

```python
class FunkSVD:
    def __init__(
        self,
        n_factors: int = 32,
        lr: float = 0.01,
        reg: float = 0.05,
        n_epochs: int = 30,
        patience: int = 3,
        seed: int = 42,
    ): ...

    def fit(
        self,
        train: pd.DataFrame,
        valid: pd.DataFrame | None = None,
    ) -> "FunkSVD":
        """
        - Инициализировать P, Q нормальным шумом (std=0.1 / sqrt(n_factors)),
          b_u, b_i нулями, mu — средним рейтингом train.
        - На каждой эпохе: shuffle пар, SGD-обновление по формулам из лекции.
        - Если valid != None — считать RMSE на valid; если он не улучшается
          patience эпох подряд, остановиться (early stopping).
        """

    def predict(self, user_id: int, item_id: int) -> float: ...

    def recommend(self, user_id: int, n: int = 10) -> list[int]: ...
```

Отдельно напишите функцию

```python
def rmse(model: FunkSVD, df: pd.DataFrame) -> float: ...
```

и короткий **скрипт эксперимента** (10–20 строк): прогнать `prepare_split` → обучить `FunkSVD` → замерить RMSE на тесте → сравнить с baseline `global_mean` (просто $\mu$). Ожидается, что FunkSVD даст RMSE **ниже** baseline хотя бы на 0.05.


In [3]:
import numpy as np
import pandas as pd

class FunkSVD:
    def __init__(
        self,
        n_factors: int = 32,
        lr: float = 0.01,
        reg: float = 0.05,
        n_epochs: int = 30,
        patience: int = 3,
        seed: int = 42,
    ):
        self.n_factors = n_factors
        self.lr = lr
        self.reg = reg
        self.n_epochs = n_epochs
        self.patience = patience
        self.seed = seed

    def fit(self, train: pd.DataFrame, valid: pd.DataFrame | None = None) -> "FunkSVD":
        np.random.seed(self.seed)

        # Создание маппингов (индексы матриц)
        self.user2idx = {u: i for i, u in enumerate(train['user_id'].unique())}
        self.item2idx = {i: idx for idx, i in enumerate(train['item_id'].unique())}
        self.idx2item = {idx: i for i, idx in self.item2idx.items()}

        n_users = len(self.user2idx)
        n_items = len(self.item2idx)

        # Инициализация параметров
        self.mu = train['rating'].mean()
        self.b_u = np.zeros(n_users)
        self.b_i = np.zeros(n_items)

        # Инициализация P и Q нормальным шумом
        std = 0.1 / np.sqrt(self.n_factors)
        self.P = np.random.normal(0, std, size=(n_users, self.n_factors))
        self.Q = np.random.normal(0, std, size=(n_items, self.n_factors))

        # История для фильтрации просмотренного в recommend
        self.user_history = train.groupby('user_id')['item_id'].apply(set).to_dict()

        # Подготовка данных для быстрого прохода (Numpy arrays)
        u_indices = np.array([self.user2idx[u] for u in train['user_id']])
        i_indices = np.array([self.item2idx[i] for i in train['item_id']])
        ratings = train['rating'].values

        # Цикл обучения (SGD)
        best_rmse = float('inf')
        patience_counter = 0
        best_weights = None

        for epoch in range(self.n_epochs):
            # Перемешивание на каждой эпохе
            idx = np.arange(len(ratings))
            np.random.shuffle(idx)
            u_indices_shuffled = u_indices[idx]
            i_indices_shuffled = i_indices[idx]
            ratings_shuffled = ratings[idx]

            # Векторизованный проход в питоне медленный, используем итерацию по 1 элементу
            # (в реальном проде для этого используют Numba или Cython)
            for u, i, r in zip(u_indices_shuffled, i_indices_shuffled, ratings_shuffled):
                # Предсказание
                pred = self.mu + self.b_u[u] + self.b_i[i] + np.dot(self.P[u], self.Q[i])
                err = r - pred

                # Обновление смещений (biases)
                self.b_u[u] += self.lr * (err - self.reg * self.b_u[u])
                self.b_i[i] += self.lr * (err - self.reg * self.b_i[i])

                # Сохраняем старое значение P_u для корректного шага Q_i
                p_u_old = self.P[u].copy()

                # Обновление скрытых факторов
                self.P[u] += self.lr * (err * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (err * p_u_old - self.reg * self.Q[i])

            # Early stopping
            if valid is not None:
                current_rmse = rmse(self, valid)
                if current_rmse < best_rmse - 1e-4:
                    best_rmse = current_rmse
                    patience_counter = 0
                    # Кэшируем лучшие веса
                    best_weights = (self.b_u.copy(), self.b_i.copy(), self.P.copy(), self.Q.copy())
                else:
                    patience_counter += 1

                if patience_counter >= self.patience:
                    # Откат к лучшим весам перед остановкой
                    if best_weights is not None:
                        self.b_u, self.b_i, self.P, self.Q = best_weights
                    break

        return self

    def predict(self, user_id: int, item_id: int) -> float:
        # Базовое предсказание = глобальное среднее
        pred = self.mu

        known_u = user_id in self.user2idx
        known_i = item_id in self.item2idx

        if known_u:
            u = self.user2idx[user_id]
            pred += self.b_u[u]
        if known_i:
            i = self.item2idx[item_id]
            pred += self.b_i[i]
        if known_u and known_i:
            pred += np.dot(self.P[u], self.Q[i])

        return pred

    def recommend(self, user_id: int, n: int = 10) -> list[int]:
        if user_id not in self.user2idx:
            return []

        u = self.user2idx[user_id]

        # Векторизованный подсчет скоров для всех товаров
        scores = self.mu + self.b_u[u] + self.b_i + self.Q.dot(self.P[u])

        # Исключение уже просмотренных
        seen = self.user_history.get(user_id, set())
        seen_indices = [self.item2idx[i] for i in seen if i in self.item2idx]
        scores[seen_indices] = -np.inf

        # Выбор top-N (используем быструю сортировку)
        top_indices = np.argpartition(-scores, n - 1)[:n]
        top_indices = top_indices[np.argsort(-scores[top_indices])]

        return [self.idx2item[idx] for idx in top_indices if scores[idx] > -np.inf]

def rmse(model: FunkSVD, df: pd.DataFrame) -> float:
    # Итератор по датафрейму (предсказываем по одной паре)
    preds = np.array([model.predict(u, i) for u, i in zip(df['user_id'], df['item_id'])])
    actuals = df['rating'].values
    return np.sqrt(np.mean((actuals - preds) ** 2))

In [4]:
# Для теста взял датасет MovieLens на 100K оценок
logs = pd.read_csv("http://files.grouplens.org/datasets/movielens/ml-100k/u.data",
                   sep='\t', names=["user_id", "item_id", "rating", "timestamp"])

# Разбиваем данные глобальным временным сплитом
train_full, test = prepare_split(logs, k_core=5, test_frac=0.2)

# Выделяем последние 10% от train_full под валидацию (для early stopping)
split_idx = int(len(train_full) * 0.9)
train = train_full.iloc[:split_idx]
valid = train_full.iloc[split_idx:]

# Инициализация и обучение FunkSVD
model = FunkSVD(n_epochs=20, lr=0.01, reg=0.05).fit(train, valid)

# Расчет метрик и сравнение с бейзлайном
funk_rmse = rmse(model, test)
baseline_rmse = np.sqrt(np.mean((test['rating'] - model.mu) ** 2))

print(f"Baseline (Global Mean) RMSE: {baseline_rmse:.4f}")
print(f"FunkSVD RMSE: {funk_rmse:.4f}")
print(f"Improvement: {baseline_rmse - funk_rmse:.4f}")

# Проверка ожидания из задания
assert baseline_rmse - funk_rmse >= 0.05, "FunkSVD не побил baseline на 0.05"

Baseline (Global Mean) RMSE: 1.1338
FunkSVD RMSE: 1.0063
Improvement: 0.1274
